# **DINO enhanced visualisation**

## **1. Importing**

In [12]:
import torch
import numpy as np
import torchvision
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader
import h5py
import random
import kagglehub
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
import matplotlib.pyplot as plt
from torchvision.datasets import OxfordIIITPet

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
from huggingface_hub import login
login()

### **1.2 DINO FLAGSHIP MODEL 40 LAYERS WOHOO**

In [23]:
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

# Ensure dtype is defined
if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

try:
    #Load Model with Memory Optimization and Eager Attention
    # We strictly need eager attention to capture the attention weights
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device).eval()

    #Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)

    num_registers = getattr(model.config, "num_register_tokens", 4)
    print(f"Model loaded. Number of register tokens identified: {num_registers}")

    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    #Verify Layer Count
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        #Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

Loading flagship model: facebook/dinov3-vit7b16-pretrain-lvd1689m
Targeting Architecture: 40 Layers, 4096 Embedding Dim


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Model loaded. Number of register tokens identified: 4
Success! Model loaded with 40 layers.


## **2. Dataset**

### **2.1 CIFAR**

In [6]:
dataset_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
loader_cifar10 = DataLoader(dataset_cifar10, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR10 samples (OOD - CIFAR-10): {len(dataset_cifar10)}")

100%|██████████| 170M/170M [00:19<00:00, 8.59MB/s]


Amount of CIFAR10 samples (OOD - CIFAR-10): 10000


In [7]:
dataset_cifar100 = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
loader_cifar100 = DataLoader(dataset_cifar100, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR100 samples (OOD - CIFAR-100): {len(dataset_cifar100)}")

100%|██████████| 169M/169M [00:19<00:00, 8.66MB/s]


Amount of CIFAR100 samples (OOD - CIFAR-100): 10000


### **2.2 SVHN**

In [8]:
dataset_svhn = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
loader_svhn = DataLoader(dataset_svhn, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of SVHN samples (Test set): {len(dataset_svhn)}")

100%|██████████| 64.3M/64.3M [00:13<00:00, 4.71MB/s]


Amount of SVHN samples (Test set): 26032


### **2.3 FOOD101**

In [9]:
dataset_food101 = torchvision.datasets.Food101(root='./data', split='test', download=True, transform=transform)
loader_food101 = DataLoader(dataset_food101, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of Food101 samples (test set): {len(dataset_food101)}")

100%|██████████| 5.00G/5.00G [04:00<00:00, 20.7MB/s]


Amount of Food101 samples (test set): 25250


### **2.4 Pokémon**

In [10]:
path = kagglehub.dataset_download("thedagger/pokemon-generation-one")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pokemon-generation-one' dataset.
Path to dataset files: /kaggle/input/pokemon-generation-one


In [11]:
transform_kaggle = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

image_root_path = os.path.join(path, 'dataset')

dataset_pokemon_gen1 = ImageFolder(
    root=image_root_path,
    transform=transform_kaggle
)

loader_pokemon_gen1 = DataLoader(
    dataset_pokemon_gen1,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print(f"\n✅ Success! Loaded {len(dataset_pokemon_gen1)} images.")
print(f"Found classes: {dataset_pokemon_gen1.classes[:5]}...")


✅ Success! Loaded 21314 images.
Found classes: ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine']...


## **2.5 Oxford-IIIT Pets**


In [13]:
dataset_pets = OxfordIIITPet(root='./data', split='test', download=True, transform=transform)
loader_pets = DataLoader(dataset_pets, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of OxfordIIITPet samples: {len(dataset_pets)}")

100%|██████████| 792M/792M [00:35<00:00, 22.3MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 11.1MB/s]


Amount of OxfordIIITPet samples: 3669


## **3. Video Generation**

In [14]:
MODEL_NAME = "facebook/dinov3-vit7b16-pretrain-lvd1689m"
IMAGE_PATH = "your_high_res_image.jpg"  # Replace with your image
OUTPUT_VIDEO = "dinov3_attention_scan.mp4"
PATCH_SIZE = 16  # Standard patch size for ViT models
BIG_PATCH_SIZE = 2  # Combine 2x2 standard patches into one "Big Patch"
STRIDE = 1  # Step size for sliding the Big Patch (in units of standard patches)
FPS = 10  # Frames per second for the video

### **3.1 Implementation**

In [15]:
import requests
import os
from PIL import Image
from io import BytesIO
import cv2

In [32]:
def get_attention_matrix(model, img_tensor, num_registers):
    """
    Extracts the self-attention map, removing CLS and Register tokens.
    """
    with torch.no_grad():
        # Pass output_attentions=True to get access to the internal attention weights
        outputs = model(img_tensor, output_attentions=True)

    # Take last layer attention: [batch, heads, tokens, tokens]
    # We select the last layer (-1)
    attentions = outputs.attentions[-1]

    # Average across all attention heads (dim 1) and remove batch dim (dim 0)
    # Shape becomes: [tokens, tokens]
    attn_avg = attentions[0].mean(dim=0)

    #   'num_registers' tokens for Registers (typically 4)
    start_idx = 1 + num_registers

    # We slice both dimensions to isolate the patch-to-patch attention
    patch_attn = attn_avg[start_idx:, start_idx:]

    return patch_attn

In [35]:
def generate_video(key, img, patch_attn, output_dir, block_size=4):
    """
    Generates the heatmap video with a variable query block size.

    :param block_size: Number of 16px patches to aggregate (e.g., 1 = 16px box, 4 = 64px box).
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    w, h = img.size
    grid_w = w // PATCH_SIZE
    grid_h = h // PATCH_SIZE

    print(f"Processing '{key}': {w}x{h} ({grid_w}x{grid_h} grid). Block Size: {block_size} ({block_size*PATCH_SIZE}px)")

    # Output filename now includes block size for clarity
    output_path = os.path.join(output_dir, f"{key}_blk{block_size}_scan.mp4")
    video = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (w * 2, h))

    attn_cpu = patch_attn.cpu().float()
    attn_grid = attn_cpu.view(grid_h, grid_w, -1)

    # Loop with dynamic block size
    # We maintain STRIDE=1 for smooth movement, but you could also parameterize this
    for y in range(0, grid_h - block_size + 1, STRIDE):
        for x in range(0, grid_w - block_size + 1, STRIDE):

            # --- Aggregation Step ---
            # Dynamic slicing based on block_size
            query_block = attn_grid[y : y + block_size, x : x + block_size, :]

            # Sum attention for the defined block
            heatmap_flat = query_block.sum(dim=(0, 1))

            # --- Visualization ---
            heatmap_2d = heatmap_flat.view(grid_h, grid_w).numpy()

            # Normalize
            heatmap_norm = (heatmap_2d - heatmap_2d.min()) / (heatmap_2d.max() - heatmap_2d.min() + 1e-8)
            heatmap_uint8 = (heatmap_norm * 255).astype(np.uint8)

            heatmap_overlay = cv2.resize(heatmap_uint8, (w, h), interpolation=cv2.INTER_NEAREST)
            heatmap_color = cv2.applyColorMap(heatmap_overlay, cv2.COLORMAP_JET)

            frame_left = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

            # Dynamic rectangle drawing
            cv2.rectangle(
                frame_left,
                (x * PATCH_SIZE, y * PATCH_SIZE),
                ((x + block_size) * PATCH_SIZE, (y + block_size) * PATCH_SIZE),
                (0, 255, 0), 3
            )

            frame_final = np.hstack([frame_left, heatmap_color])
            video.write(frame_final)

    video.release()
    print(f"Saved: {output_path}")

## **4. Running the thing**

In [29]:
IMAGE_URLS = {
    "city_4k": "https://images.unsplash.com/photo-1477959858617-67f85cf4f1df?q=80&w=2613&auto=format&fit=crop", # Urban density
    "cat_highres": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?q=80&w=2500&auto=format&fit=crop" # Object centric
}

In [27]:
def download_image(url):
    """Downloads an image from a URL and converts it to RGB."""
    print(f"Downloading from {url[:50]}...")
    # Add headers to mimic a browser and avoid 403/406 errors from Unsplash
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return img
    except Exception as e:
        print(f"Failed to download: {e}")
        raise e

In [36]:
for key, url in IMAGE_URLS.items():
    try:
        raw_img = download_image(url)

        # Max dimension constraint to avoid OOM
        MAX_DIM = 1024
        scale = min(MAX_DIM / raw_img.width, MAX_DIM / raw_img.height)
        new_w = int((raw_img.width * scale) // PATCH_SIZE * PATCH_SIZE)
        new_h = int((raw_img.height * scale) // PATCH_SIZE * PATCH_SIZE)
        img_resized = raw_img.resize((new_w, new_h))

        # --- CRITICAL FIX: Disable resizing in processor ---
        # We perform manual resizing above. We must tell the processor to use the image AS IS.
        inputs = processor(
            images=img_resized,
            do_resize=False,
            do_center_crop=False,
            return_tensors="pt"
        ).to(device)

        # Run
        attn_matrix = get_attention_matrix(model, inputs['pixel_values'], num_registers)
        generate_video(key, img_resized, attn_matrix, "videos", block_size=8)

        del inputs, attn_matrix
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Skipping {key} due to error: {e}")

Processing 'city_4k': 1024x624 (64x39 grid). Block Size: 8 (128px)
Saved: videos/city_4k_blk8_scan.mp4
Processing 'cat_highres': 1024x704 (64x44 grid). Block Size: 8 (128px)
Saved: videos/cat_highres_blk8_scan.mp4
